# 🚀 Video2Video – Lanzador Optimizado para Kaggle (GPU Dual T4 ×2)

### ⚙️ Configuración previa en la barra lateral derecha de Kaggle:
1. **Accelerator**: Selecciona **GPU T4 x2** (te otorga 32 GB VRAM combinada).
2. **Internet**: Asegúrate de que esté en **Internet on** (requiere verificación telefónica gratuita de Kaggle).
3. **Secrets**: En `Add-ons` -> `Secrets`, agrega tus claves:
   - `GHP_GITHUB_TOKEN` *(opcional si el repo es público, obligatorio si es privado)*
   - `GOOGLE_API_KEY` *(para Gemini Flash / traducción y títulos virales)*
   - `FISH_API_KEY` *(para síntesis de voz Fish Audio TTS)*

   - `HF_TOKEN` *(obligatorio para Pyannote Diarization 3.1)*

In [ ]:
# @title ⚡ Arrancar V2V + OpenReel (2x T4 + Cloudflare Tunnel)
import os, time, threading, sys, warnings, subprocess, re, urllib.request
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=SyntaxWarning)

print("=" * 60)
print("🎬 INICIANDO VIDEO2VIDEO EN KAGGLE (DUAL T4 ACCELERATOR)")
print("=" * 60)

# --- 1. DETECCIÓN DE HARDWARE MULTI-GPU ---
try:
    import torch
    gpu_count = torch.cuda.device_count()
    print(f"🔥 GPUs NVIDIA detectadas: {gpu_count}")
    for i in range(gpu_count):
        vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"   • GPU {i}: {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")
    
    if gpu_count >= 2:
        os.environ["V2V_DEVICE"] = "cuda:0"
        os.environ["V2V_DEVICE_SECONDARY"] = "cuda:1"
        print("⚡ Modo Dual-GPU activado: GPU 0 (Whisper+Diarización) | GPU 1 (RoFormer+Render)")
    elif gpu_count == 1:
        os.environ["V2V_DEVICE"] = "cuda:0"
        os.environ["V2V_DEVICE_SECONDARY"] = "cuda:0"
        print("⚡ Modo Single-GPU activado: GPU 0")
    else:
        print("⚠️ No se detectó GPU. Recuerda activar 'GPU T4 x2' en la barra lateral de Kaggle.")
except Exception as e:
    print("⚠️ Error comprobando GPUs:", e)

# --- 2. CONFIGURACIÓN Y SECRETOS DE KAGGLE ---
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_name in ["GHP_GITHUB_TOKEN", "FISH_API_KEY", "GOOGLE_API_KEY", "HF_TOKEN"]:
        try:
            val = user_secrets.get_secret(secret_name)
            if val:
                val_str = str(val).strip()
                os.environ[secret_name] = val_str
                if secret_name == "GOOGLE_API_KEY":
                    keys = [k.strip() for k in val_str.replace(";", ",").split(",") if k.strip()]
                    if len(keys) > 1:
                        print(f"🔑 Secreto cargado: GOOGLE_API_KEY ({len(keys)} API keys detectadas con rotación automática)")
                    else:
                        masked = f"...{val_str[-4:]}" if len(val_str) >= 4 else "***"
                        print(f"🔑 Secreto cargado: GOOGLE_API_KEY ({masked})")
                else:
                    masked = f"...{val_str[-4:]}" if len(val_str) >= 4 else "***"
                    print(f"🔑 Secreto cargado: {secret_name} ({masked})")
            else:
                print(f"⚠️ Secreto '{secret_name}' está vacío en Kaggle.")
        except Exception as err:
            print(f"⚠️ No se pudo leer el secreto '{secret_name}'. Recuerda activar el interruptor de este secreto en Add-ons -> Secrets para este notebook.")
    if os.getenv("HF_TOKEN"):
        try:
            from huggingface_hub import login
            login(token=os.environ["HF_TOKEN"], write_permission=False)
            print("🤗 Hugging Face autenticado con éxito.")
        except Exception:
            pass
except Exception:
    print("ℹ️ Kaggle secrets no disponibles o no configurados en Add-ons.")

GHP_TOKEN = os.getenv("GHP_GITHUB_TOKEN", "")
REPO_URL = f"https://{GHP_TOKEN}@github.com/almagomen/v2v.git" if GHP_TOKEN else "https://github.com/almagomen/v2v.git"
REPO_PATH = Path("/kaggle/working/v2v")
BRANCH_NAME = "monorepo-subtree"
BACKEND_PORT, FRONTEND_PORT = 8000, 5173
LOG_FILE = REPO_PATH / "backend.log"

# --- 3. CLONADO Y ACTUALIZACIÓN INTELIGENTE ---
if not REPO_PATH.exists():
    print("📥 Clonando repositorio en /kaggle/working/v2v...")
    subprocess.run(["git", "clone", "-q", "-b", BRANCH_NAME, REPO_URL, str(REPO_PATH)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", "HEAD", "-q"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "origin", BRANCH_NAME, "-q"], check=True)
    diff = subprocess.run(["git", "-C", str(REPO_PATH), "diff", "--name-status", "HEAD", f"origin/{BRANCH_NAME}"],
                          capture_output=True, text=True).stdout.strip()
    if diff:
        print("🚀 ¡Cambios detectados en el repo! Sincronizando con upstream...")
        subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", f"origin/{BRANCH_NAME}", "-q"], check=True)
    else:
        print("✔️ Repositorio local al día.")

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

# Sincronizar secretos en .env local para Vite y backend
env_lines = []
for k in ["GOOGLE_API_KEY", "FISH_API_KEY", "HF_TOKEN", "GHP_GITHUB_TOKEN"]:
    v = os.getenv(k)
    if v:
        env_lines.append(f"{k}={v}\n")
if env_lines:
    try:
        (REPO_PATH / ".env").write_text("".join(env_lines), encoding="utf-8")
    except Exception:
        pass

# --- 4. INSTALACIÓN DE DEPENDENCIAS ---
if 'models_loaded' not in globals():
    print("📦 Instalando herramientas y paquetes optimizados con uv...")
    subprocess.run("pip install -q uv", shell=True)
    subprocess.run("curl -fsSL https://deno.land/install.sh | sh -s -- -y >/dev/null 2>&1", shell=True)
    deno_path = "/root/.deno/bin"
    if deno_path not in os.environ["PATH"]:
        os.environ["PATH"] += f":{deno_path}"
    subprocess.run("npm install -g pnpm --silent >/dev/null 2>&1", shell=True)

    # Paquetes clave con --no-deps para velocidad máxima
    subprocess.run("uv pip install git+https://github.com/jianfch/stable-ts.git git+https://github.com/pyannote/pyannote-audio.git --no-deps -q", shell=True)
    subprocess.run("uv pip install 'yt-dlp[default]' bs-roformer-infer tinydb sentencex soundfile 'torch-audiomentations>=0.12.0' 'torchmetrics>=1.6.1' 'optuna>=4.2.0' -q", shell=True)
    subprocess.run("uv pip install openai-whisper 'pyannote-core>=6.0.1' 'pyannote-database>=6.1.1' 'pyannote-metrics>=4.0.0' 'pyannote-pipeline>=4.0.0' 'pyannoteai-sdk>=0.3.0' --no-deps -q", shell=True)
    subprocess.run("uv pip install 'opentelemetry-api>=1.34.0' 'opentelemetry-sdk>=1.34.0' 'opentelemetry-exporter-otlp>=1.34.0' 'asteroid-filterbanks>=0.4.0' 'lightning>=2.4' 'pytorch-metric-learning>=2.8.1' -q", shell=True)
    subprocess.run("uv pip install -U 'typing-extensions>=4.12.2' 'google-genai>=2.9.0' -q", shell=True)

    # Descarga binario cloudflared para túnel HTTPS externo
    cf_bin = Path("/tmp/cloudflared")
    if not cf_bin.exists():
        print("🌐 Descargando Cloudflare Tunnel...")
        subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", str(cf_bin)], check=True)
        subprocess.run(["chmod", "+x", str(cf_bin)], check=True)

    print("🛠️ Cargando modelos en GPU...")
    from src.stt import get_whisper_model
    from src.diarization import get_diarization_model
    from src.download import get_roformer_model

    globals().update({
        'shared_whisper': get_whisper_model(),
        'shared_diarization': get_diarization_model(),
        'shared_roformer': get_roformer_model(),
        'models_loaded': True
    })
    print("✅ Modelos inicializados en memoria.")
else:
    print("⚡ Modelos detectados en memoria. Saltando carga.")

# --- 5. SERVIDOR BACKEND UVICORN ---
import uvicorn
class ControlledServer(uvicorn.Server):
    def install_signal_handlers(self): return

def run_backend():
    global server
    for m in [m for m in sys.modules if m.startswith(('api', 'src'))]:
        if 'models' in m: continue
        del sys.modules[m]
    
    try:
        import typing_extensions
        if not hasattr(typing_extensions, "sentinel"):
            typing_extensions.sentinel = getattr(typing_extensions, "Sentinel", object())
    except Exception:
        pass
    from api.main import app
    app.state.whisper = globals().get('shared_whisper')
    app.state.diarization = globals().get('shared_diarization')
    app.state.roformer = globals().get('shared_roformer')

    config = uvicorn.Config(app, host="0.0.0.0", port=BACKEND_PORT, log_level="warning", loop="asyncio", timeout_keep_alive=5)
    server = ControlledServer(config=config)
    server.run()

if 'server' in globals() and server.started:
    print("🛑 Reiniciando servidor backend...")
    server.should_exit = True
    time.sleep(2)

threading.Thread(target=run_backend, daemon=True).start()

# --- 6. FRONTEND (Vite / OpenReel) ---
OPENREEL_PATH = REPO_PATH / "openreel"
WEB_PATH = OPENREEL_PATH / "apps" / "web"

# Verificar si Vite ya está respondiendo en el puerto 5173 para no reiniciarlo innecesariamente
vite_already_running = False
try:
    req = urllib.request.Request(f"http://127.0.0.1:{FRONTEND_PORT}/")
    with urllib.request.urlopen(req, timeout=1) as resp:
        if resp.status in (200, 304):
            vite_already_running = True
except Exception:
    pass

if vite_already_running and globals().get('front_proc') and globals()['front_proc'].poll() is None:
    print("⚡ OpenReel (Vite) ya está activo en el puerto 5173. Reutilizando proceso.")
    front_proc = globals()['front_proc']
else:
    print("🏗️ Preparando dependencias de OpenReel...")
    subprocess.run(["pnpm", "install", "--silent"], cwd=str(OPENREEL_PATH), check=True)
    subprocess.run(["fuser", "-k", f"{FRONTEND_PORT}/tcp"], stderr=subprocess.DEVNULL)
    
    f_env = os.environ.copy()
    f_env["FORCE_COLOR"] = "1"
    front_proc = subprocess.Popen(
        ["pnpm", "exec", "vite", "--host", "0.0.0.0", "--port", str(FRONTEND_PORT)],
        cwd=str(WEB_PATH), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=f_env
    )
    globals()['front_proc'] = front_proc

    def log_reader(pipe):
        with open(LOG_FILE, "a") as f:
            for line in iter(pipe.readline, ''):
                if line.strip():
                    f.write(f"[FRONTEND] {line.strip()}\n")
                    f.flush()

    threading.Thread(target=log_reader, args=(front_proc.stdout,), daemon=True).start()

    # --- 7. VERIFICACIÓN DE SERVICIOS (HEALTHCHECKS) ---
    print("⏳ Esperando que OpenReel (Vite) inicialice y responda en el puerto 5173...")
    vite_ready = False
    for i in range(45):
        try:
            req = urllib.request.Request(f"http://127.0.0.1:{FRONTEND_PORT}/")
            with urllib.request.urlopen(req, timeout=1) as resp:
                if resp.status in (200, 304):
                    vite_ready = True
                    print(f"✅ OpenReel Vite activo y respondiendo correctamente.")
                    break
        except Exception:
            pass
        if front_proc.poll() is not None:
            print("❌ Error: El proceso de OpenReel terminó inesperadamente.")
            if LOG_FILE.exists():
                print("📋 Últimos logs del frontend:")
                print(LOG_FILE.read_text(encoding="utf-8")[-1000:])
            break
        time.sleep(1)

# --- 8. EXPOSICIÓN VÍA CLOUDFLARE TUNNEL (PERSISTENTE Y NON-BLOCKING) ---
# Si el túnel ya está corriendo y tenemos la URL, lo preservamos para NO cambiar de dominio ni perder el timeline
cf_already_running = (
    globals().get('cf_proc') is not None
    and globals()['cf_proc'].poll() is None
    and globals().get('tunnel_url') is not None
)

if cf_already_running:
    tunnel_url = globals()['tunnel_url']
    print(f"♻️ Preservando túnel Cloudflare existente: {tunnel_url}")
    print("   (Se mantiene el mismo dominio público para conservar tu sesión y timeline en el navegador)")
else:
    print("🌐 Levantando túnel HTTPS público para acceder a OpenReel...")
    subprocess.run(["pkill", "-f", "cloudflared"], stderr=subprocess.DEVNULL)
    time.sleep(0.5)
    cf_proc = subprocess.Popen(
        ["/tmp/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{FRONTEND_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    
    tunnel_url = None
    for _ in range(40):
        line = cf_proc.stdout.readline()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break
        time.sleep(0.5)

    def cf_drainer(pipe):
        for _ in iter(pipe.readline, ''):
            pass
    threading.Thread(target=cf_drainer, args=(cf_proc.stdout,), daemon=True).start()
    
    globals()['cf_proc'] = cf_proc
    globals()['tunnel_url'] = tunnel_url

print("\n" + "=" * 60)
if tunnel_url:
    print("🎉 ¡ENTORNO LISTO Y FUNCIONANDO EN SEGUNDO PLANO!")
    print("🔗 Abre OpenReel en tu navegador:")
    print(f"👉 {tunnel_url}")
    print("=" * 60)
    print("⚡ Backend, Vite y Cloudflare Tunnel siguen activos en background.")
    print("   El kernel de Kaggle queda liberado para usar otras celdas sin bloquearse.")
else:
    print("⚠️ No se pudo obtener la URL de Cloudflare automáticamente. Revisa los logs.")
print("=" * 60)


In [ ]:
# @title 📜 Ver últimos logs (Frontend y Backend)
import os
from pathlib import Path
log_path = Path('/kaggle/working/v2v/backend.log')
if log_path.exists():
    print(log_path.read_text(encoding='utf-8')[-2000:])
else:
    print('No hay archivo de log disponible aún.')


---
## 🔬 Prueba Paso a Paso del Pipeline (Diagnóstico Función por Función)
Ejecuta las siguientes celdas de forma individual para ver los logs que genera cada función exactamente igual que en el panel de V2V, midiendo los tiempos de cada etapa.

In [ ]:
# @title 🔗 0. Configurar URL a Probar
url = "https://www.youtube.com/watch?v=C0XGm2m5Yf4"  # @param {type:"string"}
print(f"🎯 URL seleccionada: {url}")


In [ ]:
# @title 📥 1. Descarga y Separación de Stems (yt-dlp + BS-RoFormer)
import time
from src.download import yt

print(f"⚙️ Ejecutando descarga y separación de stems para: {url}\n")
t0 = time.perf_counter()
video_path, info = yt(url, vocals_separation=True)
t_step1 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 1 completado en {t_step1:.1f}s")
print(f"📁 Video descargado: {video_path}")
print(f"📁 Vocals extraídas: {video_path.with_name('vocals.wav')}")
print(f"📁 Pista instrumental: {video_path.with_name('no_vocals.wav')}")
print("=" * 60)


In [ ]:
# @title 📝 2. Transcripción (Whisper) y Diarización (Pyannote)
import time
from src.stt import transcribe_and_process

vocals_path = video_path.with_name("vocals.wav")
print(f"📝 Procesando STT + Diarización sobre: {vocals_path.name}\n")
t0 = time.perf_counter()
res_tuple = transcribe_and_process(vocals_path)
result, transcription, diarization = res_tuple[0], res_tuple[1], res_tuple[2]
t_step2 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 2 completado en {t_step2:.1f}s")
print(f"📊 Total segmentos detectados: {len(result)}")
print(f"🌐 Idioma original: {getattr(transcription, 'language', 'desconocido')}")
if result:
    print(f"🔹 Primer segmento: [{result[0]['start']:.2f}s - {result[0]['end']:.2f}s] {result[0]['speaker']}: {result[0]['text']}")
print("=" * 60)


In [ ]:
# @title 🌐 3. Traducción y Adaptación de Diálogos (Gemini)
import time
from src.translation import translate_text_list

texts_to_translate = [r["text"] for r in result]
src_lang = getattr(transcription, "language", "en")
print(f"🌐 Enviando {len(texts_to_translate)} frases a traducir ({src_lang} -> es)...\n")

t0 = time.perf_counter()
translated_texts = translate_text_list(
    texts_to_translate,
    target_lang="es",
    source_lang=src_lang,
    print_full_prompt=True
)
t_step3 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 3 completado en {t_step3:.1f}s")
if translated_texts:
    print(f"✨ Frases adaptadas con éxito: {len(translated_texts)}")
    print(f"   [Original 1]: {texts_to_translate[0]}")
    print(f"   [Adaptada 1]: {translated_texts[0]}")
else:
    print("❌ Falló la traducción.")
print("=" * 60)


In [ ]:
# @title ✨ 4. Generación de Título Viral (Gemini)
import time
from src.translation import generate_tiktok_title

# @markdown **Opciones de entrada:**
# @markdown Elige si generar el título basándote en los **diálogos** (transcripción completa) o en la **URL de YouTube**:
input_source = "dialogues"  # @param ["dialogues", "url"]
show_prompt = True  # @param {type:"boolean"}

if input_source == "dialogues":
    fallback_summary = " ".join(translated_texts) if translated_texts else " ".join([r["text"] for r in result])
    selected_url = None
else:
    fallback_summary = None
    selected_url = url

print(f"✨ Generando título viral para TikTok (fuente: {input_source})...\n")

t0 = time.perf_counter()
viral_title = generate_tiktok_title(
    youtube_url=selected_url,
    target_lang="es",
    fallback_text=fallback_summary,
    print_full_prompt=show_prompt
)
t_step4 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 4 completado en {t_step4:.1f}s")
print(f"🔥 Título viral resultante:\n👉 {viral_title}")
print("=" * 60)


In [ ]:
# @title ✂️ 5. Corte de Secuencias de Audio (Vocals EN)
import time
from src.pipeline import split_audio_sequences

# Estructurar segmentos con el texto traducido
combined_segments = []
tr_list = translated_texts if translated_texts else [r["text"] for r in result]
for orig, tr_text in zip(result, tr_list):
    seg = dict(orig)
    seg["text"] = tr_text
    combined_segments.append(seg)

print(f"✂️ Cortando {len(combined_segments)} clips de audio en Seqs_src...\n")
t0 = time.perf_counter()
split_paths = split_audio_sequences(vocals_path, combined_segments)
t_step5 = time.perf_counter() - t0

print("=" * 60)
print(f"✅ Paso 5 completado en {t_step5:.1f}s")
print(f"✂️ Archivos generados: {len(split_paths)} en {vocals_path.parent / 'Seqs_src'}")
print("=" * 60)


In [ ]:
# @title 🗣️ 6. Síntesis de Voz TTS y Ajuste de Tempo (Fish Audio + FFmpeg)
import time
from src.audio_utils import clone_voice_segments
from src.tts import synthesize_speech

speaker_audio = video_path.with_name("vocals.wav")
print(f"🗣️ Iniciando síntesis TTS para {len(combined_segments)} segmentos...\n")

t0 = time.perf_counter()
paths_es, times, rate = clone_voice_segments(
    video_path,
    combined_segments,
    "es",
    speaker_audio,
    synthesize_speech,
    slowdown_range=(0.5, 0.8),
    stretch_target_ratio=0.85
)
t_step6 = time.perf_counter() - t0

print("\n" + "=" * 60)
print(f"✅ Paso 6 completado en {t_step6:.1f}s")
print(f"🗣️ Audios generados en Seqs_trg: {len(paths_es)}")
print("=" * 60)
